In [64]:
import os
import sys
import pandas as pd

from collections import Counter
from from_root import from_root
from tqdm.auto import tqdm

sys.path.insert(0, str(from_root("src")))

from model_loading import load_model
from read_and_write_docs import read_rds, read_jsonl
from n_gram_tracing import (
    common_ngrams,
    ngram_occurrence_stats
)
from utils import apply_temp_doc_id, build_metadata_df
from greedy_string_tiling import gst_many_knowns_vs_unknown_tiles
from greedy_string_tiling_v2 import gst_many_knowns_vs_unknown_tiles_v2

In [65]:
def dedupe_ngrams(ngrams):
    """
    Deduplicate n-grams while preserving order.
    """
    return [list(x) for x in dict.fromkeys(tuple(g) for g in ngrams)]


def sort_ngrams(ngrams):
    """
    Sort first by number of tokens, then by total character length.
    """
    return sorted(
        ngrams,
        key=lambda x: (len(x), sum(len(str(token)) for token in x))
    )

In [66]:
def global_second_pass_greatest_common(
    known_texts,
    unknown_text,
    tokenizer,
    *,
    lowercase=True,
):
    all_common = []

    # First pass: collect all pairwise common n-grams
    for known_text in known_texts:
        pair_common = common_ngrams(
            text1=known_text,
            text2=unknown_text,
            tokenizer=tokenizer,
            include_subgrams=True,
            lowercase=lowercase,
        )

        all_common.extend(pair_common)

    # Problem-level candidate list
    global_common = dedupe_ngrams(all_common)
    global_common = sort_ngrams(global_common)

    if not global_common:
        return []

    unknown_stats = ngram_occurrence_stats(
        ngrams=global_common,
        text=unknown_text,
        tokenizer=tokenizer,
        lowercase=lowercase,
    )

    known_stats_list = [
        ngram_occurrence_stats(
            ngrams=global_common,
            text=known_text,
            tokenizer=tokenizer,
            lowercase=lowercase,
        )
        for known_text in known_texts
    ]

    kept = []

    for g in dict.fromkeys(tuple(x) for x in global_common):
        unknown_keep = unknown_stats.get(g, {}).get("keep", False)

        known_keep = any(
            stats.get(g, {}).get("keep", False)
            for stats in known_stats_list
        )

        if unknown_keep and known_keep:
            kept.append(list(g))

    kept = sort_ngrams(kept)
    
    return kept

In [67]:
def normalise_token_lists(token_lists):
    """
    Convert list-of-token-lists into tuple form so they can be compared/hashable.
    Example:
        [[1, 2], [3, 4]] -> [(1, 2), (3, 4)]
    """
    return [tuple(x) for x in token_lists]


def compare_token_lists(gst_common, tracing_common):
    """
    Compare GST and tracing outputs.

    Uses Counter so duplicates are handled correctly.
    Also returns readable lists for items only found in each output.
    """
    gst_norm = normalise_token_lists(gst_common)
    tracing_norm = normalise_token_lists(tracing_common)

    gst_counter = Counter(gst_norm)
    tracing_counter = Counter(tracing_norm)

    same_output = gst_counter == tracing_counter

    only_in_gst = list((gst_counter - tracing_counter).elements())
    only_in_tracing = list((tracing_counter - gst_counter).elements())

    return {
        "same_output": same_output,
        "gst_len": len(gst_norm),
        "tracing_len": len(tracing_norm),
        "num_only_in_gst": len(only_in_gst),
        "num_only_in_tracing": len(only_in_tracing),
        "only_in_gst": [list(x) for x in only_in_gst],
        "only_in_tracing": [list(x) for x in only_in_tracing],
    }

In [68]:
# --- set your args (strings) ---
base_loc = "/Volumes/BCross"

if not os.path.exists(base_loc):
    print("Using Offline Location")
    base_loc = "/Users/user/Documents/uni_work_offline"
else:
    print("Using Volume Location")

MODEL_LOC = f"{base_loc}/models/gpt2"

CORPUS_LIST = [
    "Yelp",
    "Wiki"
]

DATA_TYPE_LIST = [
    "test",
    "training"
]

Using Volume Location


In [74]:
tokenizer, model = load_model(MODEL_LOC)

In [75]:
from tqdm.auto import tqdm
import pandas as pd

comparison_rows = []

for DATA_TYPE in DATA_TYPE_LIST:

    METADATA_LOC = f"{base_loc}/datasets/author_verification/{DATA_TYPE}/metadata.rds"
    metadata = read_rds(METADATA_LOC)

    for CORPUS in CORPUS_LIST:

        KNOWN_LOC = f"{base_loc}/datasets/author_verification/{DATA_TYPE}/{CORPUS}/known_raw.jsonl"
        UNKNOWN_LOC = f"{base_loc}/datasets/author_verification/{DATA_TYPE}/{CORPUS}/unknown_raw.jsonl"

        known = read_jsonl(KNOWN_LOC)
        known = apply_temp_doc_id(known)

        unknown = read_jsonl(UNKNOWN_LOC)
        unknown = apply_temp_doc_id(unknown)

        filtered_metadata = metadata[
            metadata["corpus"] == CORPUS
        ].copy()

        agg_metadata = build_metadata_df(filtered_metadata, known, unknown)

        problems = sorted(agg_metadata["problem"].unique())

        desc = f"{DATA_TYPE} | {CORPUS}"

        for selected_problem in tqdm(problems, desc=desc):
            problem_metadata = filtered_metadata[
                filtered_metadata["problem"] == selected_problem
            ].copy()

            known_author = problem_metadata["known_author"].iloc[0]
            unknown_author = problem_metadata["unknown_author"].iloc[0]

            selected_known = known[known["author"] == known_author]
            selected_unknown = unknown[unknown["author"] == unknown_author]

            unknown_text = selected_unknown["text"].iloc[0]
            known_texts = selected_known["text"].tolist()

            gst_result = gst_many_knowns_vs_unknown_tiles_v2(
                known_texts=known_texts,
                unknown_text=unknown_text,
                tokenizer=tokenizer,
                min_match=1,
                lowercase=True,
            )

            gst_common = gst_result["tile_token_lists"]

            tracing_common = global_second_pass_greatest_common(
                known_texts,
                unknown_text,
                tokenizer,
                lowercase=True,
            )
            
            comparison = compare_token_lists(gst_common, tracing_common)

            comparison_rows.append({
                "data_type": DATA_TYPE,
                "corpus": CORPUS,
                "selected_problem": selected_problem,
                **comparison,
            })

comparison_df = pd.DataFrame(comparison_rows)

test | Yelp: 100%|██████████| 480/480 [00:12<00:00, 39.84it/s]
test | Wiki: 100%|██████████| 224/224 [00:34<00:00,  6.42it/s]
training | Yelp: 100%|██████████| 320/320 [00:07<00:00, 41.30it/s]
training | Wiki: 100%|██████████| 150/150 [00:21<00:00,  6.85it/s]


In [76]:
comparison_df

,data_type,corpus,selected_problem,same_output,gst_len,tracing_len,num_only_in_gst,num_only_in_tracing,only_in_gst,only_in_tracing
0,test,Yelp,--KQJPdrU0Md97DiOliDzw vs --KQJPdrU0Md97DiOliDzw,False,49,50,1,2,[[Ġlike]],"[[Ġlike, Ġa], [Ġbut, Ġit]]"
1,test,Yelp,--KQJPdrU0Md97DiOliDzw vs -m3ngEB9Sl5_rdYSklbt7w,False,47,47,2,2,"[['t], ['re]]","[[,, Ġthe], [Ġyou, 're]]"
2,test,Yelp,--Qh8yKWAvIP4V4K8ZPfHA vs --Qh8yKWAvIP4V4K8ZPfHA,False,51,51,1,1,[[Ġhave]],"[[Ġthey, Ġhave]]"
3,test,Yelp,--Qh8yKWAvIP4V4K8ZPfHA vs -QHl6VAjLPXgB9CNa9YD3g,False,55,55,1,1,[[Ġseemed]],"[[Ġseemed, Ġto]]"
4,test,Yelp,--kedvpjB1PT28X_gArafA vs --kedvpjB1PT28X_gArafA,False,57,57,1,1,[[Ġit]],"[[Ġit, .]]"
...,...,...,...,...,...,...,...,...,...,...
1169,training,Wiki,Haymaker vs HeadleyDown,False,143,143,7,7,"[[i], ['m], [in], [it], [this], [there], [i, 'm]]","[[Ċ, in], [,, Ġas], [Ċ, this], [Ġto, Ġdo], [th..."
1170,training,Wiki,HeadleyDown vs HeadleyDown,False,222,222,13,13,"[[it], [Ġn], [any], [Ġwhy], [Ġeng], [Ġpart], [...","[[Ġto, Ġbe], [ology, .], [Ġand, Ġis], [Ġwhy, Ġ..."
1171,training,Wiki,HeadleyDown vs Hipocrite,False,161,161,6,6,"[[i], [?], [if], [Ġwhat], [Ġbeing], [Ġon, Ġthis]]","[[Ġand, Ċ], [Ġnot, ,], [,, Ġwhat], [Ġby, Ġone]..."
1172,training,Wiki,Hipocrite vs Hipocrite,False,141,144,6,9,"[[how], [this], [Ġtalk], [there], [Ġencycloped...","[[,, Ġit], [Ċ, how], [Ċ, this], [Ċ, there], [Ġ..."


In [77]:
comparison_df[comparison_df['same_output'] == True]

,data_type,corpus,selected_problem,same_output,gst_len,tracing_len,num_only_in_gst,num_only_in_tracing,only_in_gst,only_in_tracing
5,test,Yelp,--kedvpjB1PT28X_gArafA vs -KFjONqNDuBfKDeKAoA-bg,True,51,51,0,0,[],[]
17,test,Yelp,-E4Smvof6DdBHCnZNyxNMw vs -e5VFAI-WBHBL7PFMNArDg,True,48,48,0,0,[],[]
30,test,Yelp,-KFjONqNDuBfKDeKAoA-bg vs -KFjONqNDuBfKDeKAoA-bg,True,47,47,0,0,[],[]
35,test,Yelp,-QHl6VAjLPXgB9CNa9YD3g vs -qRY3o7VLMZtdoNKXQf0MA,True,44,44,0,0,[],[]
36,test,Yelp,-St5u6epqtXK_VGlsJPnsA vs -St5u6epqtXK_VGlsJPnsA,True,43,43,0,0,[],[]
...,...,...,...,...,...,...,...,...,...,...
998,training,Yelp,4go2H9Os9mJ8_mfpnnFI0A vs 4go2H9Os9mJ8_mfpnnFI0A,True,53,53,0,0,[],[]
1001,training,Yelp,4k43P1zuES57ewaGC4jYjg vs 4k43P1zuES57ewaGC4jYjg,True,44,44,0,0,[],[]
1011,training,Yelp,4qB6lRw0cIi2mwUFbMhFCQ vs 4rGP_TDwRw3i4oPJJHFfKQ,True,42,42,0,0,[],[]
1015,training,Yelp,4rllndtbWasmhbKpa6YLww vs 4rsklgVwg7uOdaRfpHT2Vg,True,48,48,0,0,[],[]


In [78]:
comparison_df[comparison_df['same_output'] == False]

,data_type,corpus,selected_problem,same_output,gst_len,tracing_len,num_only_in_gst,num_only_in_tracing,only_in_gst,only_in_tracing
0,test,Yelp,--KQJPdrU0Md97DiOliDzw vs --KQJPdrU0Md97DiOliDzw,False,49,50,1,2,[[Ġlike]],"[[Ġlike, Ġa], [Ġbut, Ġit]]"
1,test,Yelp,--KQJPdrU0Md97DiOliDzw vs -m3ngEB9Sl5_rdYSklbt7w,False,47,47,2,2,"[['t], ['re]]","[[,, Ġthe], [Ġyou, 're]]"
2,test,Yelp,--Qh8yKWAvIP4V4K8ZPfHA vs --Qh8yKWAvIP4V4K8ZPfHA,False,51,51,1,1,[[Ġhave]],"[[Ġthey, Ġhave]]"
3,test,Yelp,--Qh8yKWAvIP4V4K8ZPfHA vs -QHl6VAjLPXgB9CNa9YD3g,False,55,55,1,1,[[Ġseemed]],"[[Ġseemed, Ġto]]"
4,test,Yelp,--kedvpjB1PT28X_gArafA vs --kedvpjB1PT28X_gArafA,False,57,57,1,1,[[Ġit]],"[[Ġit, .]]"
...,...,...,...,...,...,...,...,...,...,...
1169,training,Wiki,Haymaker vs HeadleyDown,False,143,143,7,7,"[[i], ['m], [in], [it], [this], [there], [i, 'm]]","[[Ċ, in], [,, Ġas], [Ċ, this], [Ġto, Ġdo], [th..."
1170,training,Wiki,HeadleyDown vs HeadleyDown,False,222,222,13,13,"[[it], [Ġn], [any], [Ġwhy], [Ġeng], [Ġpart], [...","[[Ġto, Ġbe], [ology, .], [Ġand, Ġis], [Ġwhy, Ġ..."
1171,training,Wiki,HeadleyDown vs Hipocrite,False,161,161,6,6,"[[i], [?], [if], [Ġwhat], [Ġbeing], [Ġon, Ġthis]]","[[Ġand, Ċ], [Ġnot, ,], [,, Ġwhat], [Ġby, Ġone]..."
1172,training,Wiki,Hipocrite vs Hipocrite,False,141,144,6,9,"[[how], [this], [Ġtalk], [there], [Ġencycloped...","[[,, Ġit], [Ċ, how], [Ċ, this], [Ċ, there], [Ġ..."
